## v3 — the tuned kernel

same algorithm, same notebook (m, r, o). four changes, each with a named cause:

1. **`exp2` not `exp`** — turing has a hardware `ex2.approx`. fold log2(e) into the scale once,
   the exponential becomes ~free. (official triton 06-fused-attention does exactly this.)
2. **`tl.dot(p, v, acc)`** — 3-arg accumulate. the add fuses into the MMA.
3. **autotune over num_stages** — this is the real fix. num_stages>1 lets triton software-pipeline
   the K/V loads so HBM latency overlaps with the dot, instead of stalling every iteration.
4. **scale = 1/sqrt(d)** — algorithm 1 has it. v2 passed 1.0. fixing for fidelity.

config space taken from triton-lang/triton `python/tutorials/06-fused-attention.py`,
pruned to BLOCK_M >= BLOCK_N as they do. plain pointer loads, not TMA descriptors —
TMA is sm_90+, this must run on the T4.

In [ ]:
import torch, triton, triton.language as tl, math
print(torch.__version__, triton.__version__, torch.cuda.get_device_name(0))
print('capability:', torch.cuda.get_device_capability(0))

In [ ]:
configs = [
    triton.Config({'BLOCK_M': BM, 'BLOCK_N': BN}, num_stages=s, num_warps=w)
    for BM in [64, 128]
    for BN in [32, 64, 128]
    for s in [2, 3, 4]
    for w in [4, 8]
    if BM >= BN
]

@triton.autotune(configs=configs, key=['N', 'D'])
@triton.jit
def flash_fwd(Q, K, V, O, N, qk_scale,
              stride_qm, stride_qd, stride_km, stride_kd,
              stride_vm, stride_vd, stride_om, stride_od,
              BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, D: tl.constexpr):
    pid = tl.program_id(0)
    offs_m = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, D)
    # residence: my Q block, loaded ONCE
    q = tl.load(Q + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd,
                mask=offs_m[:, None] < N, other=0.0)
    # notebooks
    m_i = tl.full((BLOCK_M,), float('-inf'), tl.float32)
    r_i = tl.zeros((BLOCK_M,), tl.float32)
    acc = tl.zeros((BLOCK_M, D), tl.float32)
    # the shelf parades. num_stages makes triton prefetch these loads.
    for j0 in tl.range(0, N, BLOCK_N):
        offs_n = j0 + tl.arange(0, BLOCK_N)
        k = tl.load(K + offs_n[:, None] * stride_km + offs_d[None, :] * stride_kd,
                    mask=offs_n[:, None] < N, other=0.0)
        # qk_scale already carries log2(e), so everything below lives in base-2 currency
        s = tl.dot(q, tl.trans(k)) * qk_scale
        s = tl.where(offs_n[None, :] < N, s, float('-inf'))
        m_new = tl.maximum(m_i, tl.max(s, 1))
        conv = tl.math.exp2(m_i - m_new)        # repair factor, base 2
        p = tl.math.exp2(s - m_new[:, None])    # newcomers, base 2
        r_i = conv * r_i + tl.sum(p, 1)
        v = tl.load(V + offs_n[:, None] * stride_vm + offs_d[None, :] * stride_vd,
                    mask=offs_n[:, None] < N, other=0.0)
        acc = acc * conv[:, None]
        acc = tl.dot(p.to(v.dtype), v, acc)     # 3-arg: add fuses into the MMA
        m_i = m_new
    acc = acc / r_i[:, None]
    tl.store(O + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od,
             acc.to(O.dtype.element_ty), mask=offs_m[:, None] < N)

LOG2E = 1.4426950408889634

def flash_attention(Q, K, V):
    N, d = Q.shape
    O = torch.empty_like(Q)                      # fp16 out, matches naive (v2 wrote fp32)
    qk_scale = (1.0 / math.sqrt(d)) * LOG2E      # scale AND base-2 conversion, once
    grid = lambda META: (triton.cdiv(N, META['BLOCK_M']),)
    flash_fwd[grid](Q, K, V, O, N, qk_scale,
                    Q.stride(0), Q.stride(1), K.stride(0), K.stride(1),
                    V.stride(0), V.stride(1), O.stride(0), O.stride(1),
                    D=d)
    return O

### exactness first — a faster wrong kernel is worthless

In [ ]:
def naive_attention(Q, K, V):
    S = (Q @ K.T) * (1.0 / math.sqrt(Q.shape[1]))   # now scaled, matching Algorithm 1
    P = torch.softmax(S.float(), dim=-1).to(Q.dtype)
    return P @ V

torch.manual_seed(0)
N, d = 1024, 64
mk = lambda: torch.randn(N, d, device='cuda', dtype=torch.float16)
Q, K, V = mk(), mk(), mk()
ref, out = naive_attention(Q, K, V).float(), flash_attention(Q, K, V).float()
print('max abs diff:', (ref - out).abs().max().item())
assert torch.allclose(ref, out, atol=2e-2), 'exactness failed'
print('exactness: PASS (fp16 in, fp32 accum, exp2 path)')
print('autotune picked:', flash_fwd.best_config)

### the three receipts, re-measured

note the autotune warm-up is excluded from timing by the 3 untimed calls in `bench`.

In [ ]:
import time
def bench(fn, *args, iters=30):
    for _ in range(5): fn(*args)          # warm-up ALSO burns in the autotune search
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(iters): fn(*args)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3

def peak_mem(fn, *args):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    fn(*args); torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 2**20

print(f'{"N":>6} {"naive ms":>9} {"flash ms":>9} {"ratio":>7} {"naive MiB":>10} {"flash MiB":>10}  config')
results = []
for N in [1024, 2048, 4096, 8192, 16384, 32768, 65536]:
    mk = lambda: torch.randn(N, d, device='cuda', dtype=torch.float16)
    Q, K, V = mk(), mk(), mk()
    tf = bench(flash_attention, Q, K, V); mf = peak_mem(flash_attention, Q, K, V)
    cfg = flash_fwd.best_config
    try:
        tn = bench(naive_attention, Q, K, V); mn = peak_mem(naive_attention, Q, K, V)
        results.append((N, round(tn,3), round(tf,3)))
        print(f'{N:>6} {tn:>9.3f} {tf:>9.3f} {tn/tf:>6.2f}x {mn:>10.0f} {mf:>10.0f}  {cfg}')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        results.append((N, None, round(tf,3)))
        print(f'{N:>6} {"OOM":>9} {tf:>9.3f} {"—":>7} {"> 15000":>10} {mf:>10.0f}  <- naive DIED; flash sails on')
    del Q, K, V; torch.cuda.empty_cache()
print()
print('paste into widget6-benchmark.html:')
print('const DATA =', [[n, a, b] for n, a, b in results], ';')

### control experiment — is it the kernel, or is it the empty grid?

single-head N x d leaves the GPU starved at small N: grid = N/BLOCK_M programs on 40 SMs.
real transformers run batch x heads of these at once. this cell isolates that variable:
same N, but 12 independent heads, launched as 12 sequential calls on both sides.
if the ratio improves a lot here, grid-filling was the story. if it barely moves,
the constant factor is the kernel itself and option (a) stands.

In [ ]:
H = 12
print(f'{"N":>6} {"naive ms":>9} {"flash ms":>9} {"ratio":>7}   (x{H} heads)')
for N in [1024, 4096, 16384]:
    mk = lambda: torch.randn(N, d, device='cuda', dtype=torch.float16)
    qs = [(mk(), mk(), mk()) for _ in range(H)]
    def run_flash():
        for q_, k_, v_ in qs: flash_attention(q_, k_, v_)
    def run_naive():
        for q_, k_, v_ in qs: naive_attention(q_, k_, v_)
    tf = bench(run_flash); tn = bench(run_naive)
    print(f'{N:>6} {tn:>9.3f} {tf:>9.3f} {tn/tf:>6.2f}x')
    del qs; torch.cuda.empty_cache()